In [1]:
%pip install datasets langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


# Dataset Loading

- [Naver 경제, IT 뉴스기사 요약 데이터셋](https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko)

In [2]:
from datasets import load_dataset
import pandas as pd

# Hugging Face 데이터셋 로드
dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")
dataset


README.md:   0%|          | 0.00/787 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\lang_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--daekeun-ml--naver-news-summarization-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv:   0%|          | 0.00/66.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/7.45M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/8.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2466 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2740 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [4]:
trainset = dataset['train']
print(type(trainset))
trainset

<class 'datasets.arrow_dataset.Dataset'>


Dataset({
    features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
    num_rows: 22194
})

In [5]:
df_train = trainset.to_pandas()
df_train.shape

(22194, 7)

In [6]:
df_train.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-05 18:01:01,IT과학,동아사이언스,한국인 생활 변화시킨 의사 마지막까지 영향력 컸던 과학자 이호왕 고려대 명예교수 소천,한탄바이러스 발견 노벨상 유력 후보로 자주 거론 한국을 대표하는 의학자이자 미생물학...,https://n.news.naver.com/mnews/article/584/000...,이 이호왕 고려대 명예교수는 바이러스의 병원체와 진단법 백신까지 모두 개발한 한국을...
3,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
4,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...


In [7]:
df = df_train.query("category=='economy'").reset_index(drop=True)
df.shape

(17088, 7)

# 데이터셋 만들기
- 뉴스기사 제목과 뉴스기사를 이용해 그 기사에 영향을 받는 주가종목을 추론하는 모델을 만든다.
- 데이터 구성
  - **입력**: 뉴스 기사 제목, 뉴스 기사
  - **출력**: 뉴스기사가 주식에 영향을 주는지 여부, 부정적인 영향을 받는 회사와 이유, 긍정적인 영향을 받는 회사와 이유, 뉴스기사 요약
- Label을 LLM을 이용해 생성한다.
  - LLM을 이용해 데이터셋을 만든 이후 그 결과를 눈으로 검토해야 한다.

## Dataset 생성 Chain 구성

In [8]:
from tqdm import tqdm
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
template = '''# Instruction
당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.
사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.

## 분석 기준
1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.
2. 영향을 준다면 다음 항목을 출력하세요.
   - `"is_stock_related": true`
   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**
   - 뉴스에 **부정적** 영향을 받는 **회사이름들**
   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**
     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.
     - `None`, 유추, 추정, 일반 논평 금지합니다.
   - 뉴스 요약 (3줄 이내)
3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.
   - `"is_stock_related": false`
   - 뉴스 요약 (3줄 이내)

# 입력 데이터(뉴스기사)

{input}


# 출력 지시사항 (Output Indicator)

- {format_instructions}

## 출력 조건:
- 뉴스에 영향을 받은 회사들은 **반드시 한국 증시에 상장된 종목** 이어야 합니다.
- 뉴스에 있는 내용만 출력결과에 포함시킵니다.
- 긍정/부정 종목은 실제 뉴스기사에 영향을 받는 회사들만 포함하세요.
- 모든 문자열은 큰따옴표(`"`)로 감쌉니다.
- 문자열 안에 따옴표가 필요하면 작은따옴표(`'`)를 사용합니다.
- 모든 키(Key)는 출력 지시사항에 명시된 property들과 정확히 일치해야 합니다.
- `"positive_reasons"` 및 `"negative_reasons"`의 값은 `None`이 될 수 없습니다.
- json format을 잘 지켜 응답데이터를 만듭니다. 배열이나 object의 마지막 항목 뒤에 `,` 를 붙이지 마세요.
- 오직 유효한 JSON 문자열(UTF-8, RFC8259 준수)만 출력합니다.
- 절대 다른 텍스트, 주석, 설명, 코드 블록 표기(```), 또는 따옴표 외의 문자열을 추가하면 안 됩니다.

## 출력 예시 (Examples)

### 뉴스가 특정 주식종목들에 **긍정적 영향이 주는 경우**:
{{'is_stock_related': true,
 'negative_reasons': [],
 'negative_stocks': [],
 'positive_reasons': [{{'세라젬': '루게릭병 환우 지원 캠페인 후원과 의료가전 지원 등 사회공헌활동을 통해 기업 이미지와 브랜드 가치가 긍정적으로 부각됨'}}],
 'positive_stocks': ['세라젬'],
 'summary': '세라젬이 루게릭병 환우를 위한 아이스버킷 챌린지 런 행사를 후원하며 의료가전과 건강기능식품 등을 지원했다. 캠페인은 루게릭병 환우 지원과 기부 문화 확산을 목표로 한다. 세라젬은 다양한 사회공헌활동을 지속하고 있다.'
}}

### 뉴스의 내용이 특정 주식종목들에 **부정적 영향이 주는 경우**:
{{
    "is_stock_related": true,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [
        "포스코",
        "현대제철"
    ],
    "negative_reasons": [
        {{"포스코": "정부가 수입규제국 조사에 적극 대응하고 비관세장벽 해소를 위해 민관 협력 강화 방침을 밝혀 철강 분야에서 수출 피해 최소화 기대"}},
        {{"현대제철": "철강·금속 품목에 대한 수입규제 대응 강화로 불합리한 무역제한 조치 개선 가능성이 높아져 수출 환경 개선 기대"}}
    ],
    "summary": "산업부는 수입 규제국의 조사에 대응하고 비관세장벽 해소를 위한 협의를 진행했다. 규제 대상 국가는 26개국, 건수는 199건에 달한다."
}}

### **뉴스기사가 주식 종목과 관련 없는 경우**:
{{
    "is_stock_related": false,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [],
    "negative_reasons": [],
    "summary": "정황근 농림축산식품부 장관이 단순가공식품 부가가치세 면제 시행 상황을 점검했다. 된장, 고추장 코너를 방문하며 현장을 살폈다."
}}'''

In [10]:
class SummarySchema(BaseModel):
    is_stock_related: bool = Field(description="한국 주식과 관련있는 뉴스인지 여부")
    positive_stocks: list[str] = Field(description="뉴스기사에 긍정적인 영향을 받는 회사들의 이름들.")
    positive_reason: list[dict[str,str]] = Field(description='뉴스내용 중 positive_stocks에 있는 각 회사들에 긍정적 영향을 주는 내용. {"회사이름":"긍정적인 이유"}')
    negative_stocks: list[str] = Field(description="뉴스기사에 부정적인 영향을 받는 회사들의 이름들.")
    negative_reason: list[dict[str,str]] = Field(description='뉴스내용 중 negative_stocks에 있는 각 회사들에 부정적 영향을 주는 내용. {"회사이름":"부정적인 이유"}')
    summary: str = Field(description="뉴스기사 요약")

parser = JsonOutputParser(pydantic_object=SummarySchema)

prompt_template = ChatPromptTemplate(
    messages=[
        ("user", template), 
    ],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model_name = "gpt-4.1"
chain = prompt_template | ChatOpenAI(model=model_name) | parser

### 개별 데이터로 Chain 테스트

In [14]:
from pprint import pprint

news_idx = 120
sample_news = df['title'].loc[news_idx]+"\n"+df['document'].loc[news_idx]
print(sample_news)

대상그룹 헌혈 캠페인 전국민 동참 레드챌린지
서울 연합뉴스 신현우 기자 5일 오전 서울 종로구 대상그룹 본사에서 이 회사 임직원들이 혈액 수급 안정 유도와 헌혈 참여 독려를 위한 전국민 동참 레드챌린지 캠페인을 홍보하고 있다. 대상 그룹은 7월 한 달간 전 국민을 대상으로 이 캠페인을 전개한다.


In [15]:
# 4.1 결과
response = chain.invoke(input={"input":sample_news})

pprint(response)

{'is_stock_related': True,
 'negative_reason': [],
 'negative_stocks': [],
 'positive_reason': [{'대상': '대상그룹이 헌혈 캠페인 등 사회공헌 활동을 전개한다는 내용이 보도되어 기업 이미지 및 '
                            '사회적 책임 경영이 긍정적으로 부각됨'}],
 'positive_stocks': ['대상'],
 'summary': '대상그룹 직원들이 국민 헌혈 동참을 위한 레드챌린지 캠페인을 홍보했다. 7월 한 달간 전 국민 대상 캠페인이 '
            '진행된다. 혈액 수급 안정과 헌혈 참여를 독려하고 있다.'}


## 답(label) 만들기

1. 뉴스제목(title)과 뉴스기사(document)를 합쳐서 입력데이터를 만든다.
2. 2의  입력데이터를 LLM에 요청해서 답변을 받은 뒤 DataFrame에 추가한다.

In [16]:
# 1. K개 샘플링

sample_nums = 100  
sample_df = df.sample(sample_nums).reset_index(drop=True)
sample_df.shape

(100, 7)

In [18]:
# 2. 뉴스제목(title)과 뉴스기사(document)를 합쳐서 프롬프트를 생성한다.

articles = sample_df['title']+"\n"+sample_df['document']
articles.shape, type(articles)

((100,), pandas.core.series.Series)

In [19]:
articles

0     국토부 세종시 부적격 당첨자 계약 취소·주택 환수 등 엄중조치\n국토교통부 청사. ...
1     코스피 하락 마감\n서울 뉴시스 김금보 기자 1일 오전 서울 중구 하나은행 딜링룸에...
2     이재용구광모 회장이 꽂힌 전장사업…삼성·LG 대격돌\n500조 시장 공략 박차…미래...
3     장마특수 제습기 수요 증가\n서울 뉴스1 송원영 기자 올해 장마철 강풍과 함께 폭우...
4     사회적기업·협동조합 한자리…대한민국 사회적경제 박람회 개최\n기사내용 요약 8 10...
                            ...                        
95    신한카드 국내 최초 민간데이터댐 ‘그랜데이터’ 세미나 개최\nSK텔레콘·KCB·공공...
96    만평 최저임금 9620원인데···\n민주노총 반발···소상공인연합회는 불만 2023...
97    코웨이 인터브랜드 베스트 코리아 브랜드 8년 연속 선정\n대한민국 Top 50 브랜...
98    송옥렬 공정거래위원장 후보자 기자간담회\n송옥렬 공정거래위원장 후보자가 5일 오후 ...
99    한은이 인터넷망 클라우드 임차한 이유는\n원격근무 환경 개선 제2의 코로나19 대비...
Length: 100, dtype: object

In [ ]:
# 3. LLM에 label 생성 요청

label_list = articles.apply(lambda x : chain.invoke({'input':x}))

In [21]:
articles[0]

'국토부 세종시 부적격 당첨자 계약 취소·주택 환수 등 엄중조치\n국토교통부 청사. 연합뉴스 서울경제 국토교통부는 세종시 부적격 청약 당첨자 등에 대한 위법행위 여부를 철저히 조사하고 엄중조치하겠다고 6일 밝혔다. 감사원은 ‘세종시 이전기관 종사자 주택 특별공급’에 대한 감사를 진행할 결과 45건 76명 의 감사결과를 확정했다. 특별공급 대상기관에 파견 근무 중인 지자체 공무원이 특별공급을 받기 위해 확인서를 위조해 공급 받거나 이전에 행복도시에서 특별공급이나 일반공급에 당첨돼 특공 대상에서 제외됐음에도 검증 부실로 공급 받은 사례 등이 적발됐다. 국토부는 감사원 감사결과에 따른 후속조치를 강도 높게 추진할 계획이다. ‘주택공급에 관한 규칙’을 위반해 청약에 당첨된 76명에 대해 세종시에 관련 사실을 통보하고 사실관계를 파악해 계약 취소 및 주택 환수 조치를 한다는 방침이다. 또 관련 법령 위반 여부를 판단하고 위법성이 확인되면 고발한다. 원희룡 국토부 장관은 “젊은 세대에게 상대적 박탈감을 준 부적격 특별공급에 대해 철저히 조사하고 계약취소 및 주택환수 형사고발까지 필요한 모든 조치를 취하겠다”고 말했다.'

In [22]:
label_list[0]

{'is_stock_related': False,
 'positive_stocks': [],
 'positive_reasons': [],
 'negative_stocks': [],
 'negative_reasons': [],
 'summary': '국토교통부가 세종시 부적격 청약 당첨자에 대해 계약 취소와 주택 환수 등 엄중조치를 예고했다. 감사원 감사 결과 위반 사례가 다수 적발됐다. 위법 시 형사고발도 추진할 방침이다.'}

In [23]:
# Label 타입 변환:  Dictionary 를 str로 변환.

label_list_str = label_list.apply(lambda x : str(x))
type(label_list_str[0]), label_list_str[0]

(str,
 "{'is_stock_related': False, 'positive_stocks': [], 'positive_reasons': [], 'negative_stocks': [], 'negative_reasons': [], 'summary': '국토교통부가 세종시 부적격 청약 당첨자에 대해 계약 취소와 주택 환수 등 엄중조치를 예고했다. 감사원 감사 결과 위반 사례가 다수 적발됐다. 위법 시 형사고발도 추진할 방침이다.'}")

In [24]:
# Label을 데이터 프레임에 추가.
sample_df['label'] = label_list_str
sample_df.head()

,date,category,press,title,document,link,summary,label
0,2022-07-06 16:54:03,economy,서울경제,국토부 세종시 부적격 당첨자 계약 취소·주택 환수 등 엄중조치,국토교통부 청사. 연합뉴스 서울경제 국토교통부는 세종시 부적격 청약 당첨자 등에 대...,https://n.news.naver.com/mnews/article/011/000...,6일 국토교통부는 ‘세종시 이전기관 종사자 주택 특별공급’에 대한 감사를 진행한 결...,"{'is_stock_related': False, 'positive_stocks':..."
1,2022-07-01 16:19:27,economy,뉴시스,코스피 하락 마감,서울 뉴시스 김금보 기자 1일 오전 서울 중구 하나은행 딜링룸에서 딜러들이 업무를 ...,https://n.news.naver.com/mnews/article/003/001...,코스피가 장중 2291.49까지 추락하며 한때 2300선 밑으로 내려가는 등 230...,"{'is_stock_related': True, 'positive_stocks': ..."
2,2022-07-05 16:19:01,economy,아이뉴스24,이재용구광모 회장이 꽂힌 전장사업…삼성·LG 대격돌,500조 시장 공략 박차…미래 먹거리 위해 전자 계열사 역량 총동원 삼성과 LG가 ...,https://n.news.naver.com/mnews/article/031/000...,삼성전자와 LG가 이재용 부회장과 구광모 회장이 차세대 성장 동력으로 점찍은 전장 ...,"{'is_stock_related': True, 'positive_stocks': ..."
3,2022-07-03 13:10:56,economy,뉴스1,장마특수 제습기 수요 증가,서울 뉴스1 송원영 기자 올해 장마철 강풍과 함께 폭우가 내리면서 유통업계도 장마 ...,https://n.news.naver.com/mnews/article/421/000...,올해 장마철 강풍과 함께 폭우가 내리면서 습도가 높아지면서 실내가 눅눅해진 탓에 제...,"{'is_stock_related': True, 'positive_stocks': ..."
4,2022-07-06 16:10:26,economy,뉴시스,사회적기업·협동조합 한자리…대한민국 사회적경제 박람회 개최,기사내용 요약 8 10일 경주 화백컨벤션센터서 열려 사회적경제 일자리 창출 등 발전...,https://n.news.naver.com/mnews/article/003/001...,제기업과 협동조합 마을·자활기업 소셜벤처 등 사회적경제계가 한자리에 모여 정보를 공...,"{'is_stock_related': False, 'positive_stocks':..."


In [25]:
############ pickle로 저장
import os
os.makedirs("dataset", exist_ok=True)

output_file = "dataset/sample_df.pkl"

sample_df.to_pickle(
    output_file
)

# import pandas as pd
# sample_df = pd.read_pickle(output_file)

# 허깅페이스에 업로드

In [26]:
from huggingface_hub import login
from datasets import load_dataset

from dotenv import load_dotenv

load_dotenv()

True

In [27]:
import os
login(os.getenv("HUGGINGFACE_API_KEY"))

In [29]:
from datasets import Dataset

# Dataset으로 변환.
dataset = Dataset.from_pandas(sample_df)	# DataFrame을 Dataset으로 변환
dataset

Dataset({
    features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
    num_rows: 100
})

In [35]:
dataset[0]

{'date': '2022-07-06 16:54:03',
 'category': 'economy',
 'press': '서울경제 ',
 'title': '국토부 세종시 부적격 당첨자 계약 취소·주택 환수 등 엄중조치',
 'document': '국토교통부 청사. 연합뉴스 서울경제 국토교통부는 세종시 부적격 청약 당첨자 등에 대한 위법행위 여부를 철저히 조사하고 엄중조치하겠다고 6일 밝혔다. 감사원은 ‘세종시 이전기관 종사자 주택 특별공급’에 대한 감사를 진행할 결과 45건 76명 의 감사결과를 확정했다. 특별공급 대상기관에 파견 근무 중인 지자체 공무원이 특별공급을 받기 위해 확인서를 위조해 공급 받거나 이전에 행복도시에서 특별공급이나 일반공급에 당첨돼 특공 대상에서 제외됐음에도 검증 부실로 공급 받은 사례 등이 적발됐다. 국토부는 감사원 감사결과에 따른 후속조치를 강도 높게 추진할 계획이다. ‘주택공급에 관한 규칙’을 위반해 청약에 당첨된 76명에 대해 세종시에 관련 사실을 통보하고 사실관계를 파악해 계약 취소 및 주택 환수 조치를 한다는 방침이다. 또 관련 법령 위반 여부를 판단하고 위법성이 확인되면 고발한다. 원희룡 국토부 장관은 “젊은 세대에게 상대적 박탈감을 준 부적격 특별공급에 대해 철저히 조사하고 계약취소 및 주택환수 형사고발까지 필요한 모든 조치를 취하겠다”고 말했다.',
 'link': 'https://n.news.naver.com/mnews/article/011/0004073351?sid=101',
 'summary': '6일 국토교통부는 ‘세종시 이전기관 종사자 주택 특별공급’에 대한 감사를 진행한 결과 특별공급을 받기 위해 확인서를 위조해 공급 받거나 이전에 행복도시에서 특별공급이나 일반공급에 당첨돼 특공 대상에서 제외됐음에도 검증 부실로 공급 받은 사례 등 ‘세종시 부적격 청약 당첨자’ 76명 의 감사결과를 확정하고 위법행위 여부를 철저히 조사하고 엄중조치하겠다고 밝혔다.',
 'label': "{'is_stock_related': Fa

In [36]:
# train/valid/test set으로 분리
dataset_dict = dataset.train_test_split(test_size=0.1)
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 90
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 10
    })
})

In [38]:
# 데이터셋을 Huggingface hub 에 업로드.
dataset_id = "naver_economy_news_stock_instruct_dataset-100"
dataset_dict.push_to_hub(dataset_id)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/mingyu-oo/naver_economy_news_stock_instruct_dataset-100/commit/5a83ea8b8df477a7fee5caf6fe264440c4ed38cd', commit_message='Upload dataset', commit_description='', oid='5a83ea8b8df477a7fee5caf6fe264440c4ed38cd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mingyu-oo/naver_economy_news_stock_instruct_dataset-100', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mingyu-oo/naver_economy_news_stock_instruct_dataset-100'), pr_revision=None, pr_num=None)

In [39]:
# Dataset load

load_data = load_dataset("mingyu-oo/naver_economy_news_stock_instruct_dataset-100")

README.md:   0%|          | 0.00/615 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\lang_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--mingyu-oo--naver_economy_news_stock_instruct_dataset-100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is

train-00000-of-00001.parquet:   0%|          | 0.00/202k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/48.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10 [00:00<?, ? examples/s]

In [42]:
d_id = "kgmyh/naver_economy_news_stock_instruct_dataset"
dataset = load_dataset(d_id)

In [43]:
dataset

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 1350
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary', 'label'],
        num_rows: 150
    })
})

매우 예리한 지적이야.
이 프로그램이 역으로 수도권 유입을 자극할 가능성도 충분히 존재하기 때문에,
그에 대한 대응 설계가 필요해. 아래에 정리해줄게.

---

✅ 문제 정의: 역유입 리스크
사용자가 지방을 알아보던 중

“아, 그래도 서울/수도권이 낫겠네”라고 결론 낼 경우,
이 프로그램이 오히려 수도권 선호 재강화 효과를 낼 수도 있음.

---

✅ 원인 분석
| 요인       | 내용                                   |
| -------- | ------------------------------------ |
| 상대 비교 효과 | 추천된 지방의 주거비·교통·문화시설을 보고 수도권과 비교하게 됨  |
| 데이터 편향   | 수도권 정보가 더 풍부하거나 세련되게 표현될 경우 인식 왜곡    |
| 사례 불균형   | 수도권 성공사례가 많고, 로컬 사례가 생략되면 설득력 약화     |
| 실행 거리감   | 정착 실행 경로가 막연하면 ‘그래도 서울에 있는 게 낫지’로 귀결 |

---

✅ 대응 방안: 설계 개선 포인트
비교 대상으로 수도권은 제외
사용자가 입력한 조건과 유사한 지방 대안만 노출
수도권 도시는 비교 옵션에서 제거하거나 제한

비교 대신 가능성 중심 서사 설계
주거비/연봉 등 단순 수치 비교보다“당신이 이곳에서 무엇을 할 수 있는지”에 초점 맞춤
서울보다 저렴하다는 말보다,“이 예산으로 가능하다”는 메시지 강조

지방 정착의 상대 이점 시뮬레이션 추가
정착 이후 커뮤니티, 사업 기회, 문화적 여유 등수도권에서 얻기 어려운 가치를 별도로 정리해 제시

이탈 리스크 사용자 행동 탐지
사용자가 "서울/경기" 키워드를 입력하거나
추천 지역을 거부하면 →로컬 성공 사례 + 커뮤니티 후속 대화 유도

수도권 대비 매력 지역 추천은 후순위
만약 수도권 생활권을 원하더라도,
인접 지방(예: 인천 대신 안성, 서울 대신 원주)처럼“완충지대” 역할 도시로 유도

---

✅ 시나리오 예시
사용자: “전 여전히 교통이 불편할 것 같아요.”
>
AI 응답: “맞아요. 그래서 OO시는 서울 강남까지 KTX로 57분이면 도착할 수 있고,
주중에는 원격 근무를 병행하면서 주말에 지역 커뮤니티에 참여하는 방식도 실제 사례로 있어요.”

---

✅ 문서화 방향 제안 (신청서 반영)
"역유입 방지를 위한 설계 고려사항" 항목을 문서 후반부에 추가
→ 사용자 분석을 기반으로 한 구조적 설계임을 강조
→ 지역 균형 발전이라는 과제 목적과의 일치성 강화

---

원한다면 바로 문서에 추가 정리해서 반영해줄게.
추가 문구나 항목 제안할까?